# 밀도맵 카운팅 (Density-map Counting) — Global Wheat Detection (PyTorch) — Colab

객체 박스 대신 **밀도맵(density map)** 을 회귀해 개수를 세는 카운팅 기본 예제입니다.

- 핵심 흐름: 각 객체 중심에 **가우시안**을 찍어 density map 생성 → CNN이 그 맵을 **회귀** → **합 = 개수**.
- IOAI 2025 **Chicken Counting** 과제의 기반 기법입니다. (공식 해법도 U-Net+density map 사용)
- [Global Wheat Detection](https://www.kaggle.com/c/global-wheat-detection) 의 밀 이삭 박스를 카운팅 대상으로 사용합니다.
- 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ **GPU 권장**. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "pillow", "scipy", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Global Wheat 데이터 다운로드

> 약 600MB라 수 분 걸릴 수 있습니다.

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("global-wheat-detection", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "train")
print("train images:", len(os.listdir(TRAIN_DIR)))

## 2. 밀도맵 생성

`train.csv` 의 bbox 중심을 이미지(→`SIZE`로 축소)에 찍고 가우시안 필터를 적용해 density map 을 만듭니다. density map 의 **합 = 박스 개수**가 되도록 정규화됩니다.

In [ ]:
import numpy as np, pandas as pd, ast, glob
from PIL import Image
from scipy.ndimage import gaussian_filter
import torch

SIZE = 256; SRC = 1024
SCALE = 100.0   # 밀도값을 키워 0으로 붕괴하는 것을 방지 (count = sum / SCALE)
df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
df["bbox"] = df["bbox"].apply(ast.literal_eval)
boxes_by_img = df.groupby("image_id")["bbox"].apply(list).to_dict()
image_ids = list(boxes_by_img.keys())

def density_map(bboxes):
    m = np.zeros((SIZE, SIZE), dtype="float32")
    s = SIZE / SRC
    for x, y, w, h in bboxes:
        cx = int((x + w/2) * s); cy = int((y + h/2) * s)
        if 0 <= cy < SIZE and 0 <= cx < SIZE:
            m[cy, cx] += 1.0
    return gaussian_filter(m, sigma=2) * SCALE   # 합 = (박스수)*SCALE

print("annotated images:", len(image_ids), "| 예: 박스수", len(boxes_by_img[image_ids[0]]))

## 3. Dataset / DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class WheatDensity(Dataset):
    def __init__(self, ids):
        self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        img = Image.open(os.path.join(TRAIN_DIR, iid + ".jpg")).convert("RGB").resize((SIZE, SIZE))
        x = np.asarray(img, dtype="float32").transpose(2,0,1) / 255.0
        dm = density_map(boxes_by_img[iid])
        return torch.from_numpy(x), torch.from_numpy(dm)[None], len(boxes_by_img[iid])

rng = np.random.RandomState(42); ids = image_ids.copy(); rng.shuffle(ids); cut = int(len(ids)*0.9)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(WheatDensity(ids[:cut]), batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(WheatDensity(ids[cut:]), batch_size=8, num_workers=2)
print("device:", device, "| train:", cut, "val:", len(ids)-cut)

## 4. 밀도 회귀 CNN (encoder-decoder)

In [ ]:
import torch.nn as nn

class DensityNet(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU())
        self.enc = nn.Sequential(blk(3,32), nn.MaxPool2d(2), blk(32,64), nn.MaxPool2d(2), blk(64,128))
        self.dec = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), blk(128,64),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), blk(64,32),
            nn.Conv2d(32,1,1), nn.ReLU(),
        )
    def forward(self, x): return self.dec(self.enc(x))

model = DensityNet().to(device)
print("params:", sum(p.numel() for p in model.parameters()))

## 5. 학습 (density MSE)

예측 맵과 정답 맵의 MSE 로 학습합니다. 카운트는 맵의 합으로 얻습니다.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
mse = nn.MSELoss()
EPOCHS = 15
for epoch in range(1, EPOCHS+1):
    model.train(); run=0; n=0
    for x, dm, cnt in train_loader:
        x, dm = x.to(device), dm.to(device)
        pred = model(x)
        loss = mse(pred, dm)
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item()*x.size(0); n += x.size(0)
    print(f"epoch {epoch:2d} | density MSE(x1000) {run/n:.4f}")

## 6. 카운트 평가 (예측 맵 합 vs 실제 개수)

In [ ]:
model.eval(); preds=[]; trues=[]
with torch.no_grad():
    for x, dm, cnt in val_loader:
        p = model(x.to(device)).sum(dim=(1,2,3)).cpu().numpy() / SCALE   # 스케일 복원
        preds.extend(p.tolist()); trues.extend(cnt.numpy().tolist())
preds = np.array(preds); trues = np.array(trues)
mae = np.abs(preds - trues).mean()
print(f"count MAE {mae:.2f} | 예측합 평균 {preds.mean():.1f} vs 실제 평균 {trues.mean():.1f}")

## 7. 시각화 (이미지 / 정답 density / 예측 density)

In [ ]:
import matplotlib.pyplot as plt
x, dm, cnt = next(iter(val_loader))
with torch.no_grad(): pred = model(x.to(device)).cpu()
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(x[0].permute(1,2,0).numpy()); ax[0].set_title(f"image (true {int(cnt[0])})")
ax[1].imshow(dm[0,0], cmap="jet"); ax[1].set_title(f"GT density (count {dm[0].sum()/SCALE:.1f})")
ax[2].imshow(pred[0,0], cmap="jet"); ax[2].set_title(f"pred density (count {pred[0].sum()/SCALE:.1f})")
for a in ax: a.axis("off")
plt.show()

## 🏁 제출 안내

이 노트북은 **밀도맵 카운팅** 연습용 데모입니다(개수 추정; 별도 제출 리더보드 없음).

- 데이터 출처: **[Global Wheat Detection](https://www.kaggle.com/c/global-wheat-detection)**

> IOAI 2025 *Chicken Counting* 과제의 기반 기법(밀도맵 회귀 카운팅)입니다.